# Phase 6f: Curvature Tensor Auto-Derivation

Metric 함수 하나로부터 JAX autodiff를 사용하여 곡률 텐서를 자동 계산한다.

$$\Gamma^\sigma_{\mu\nu} = \frac{1}{2}g^{\sigma\rho}\left(\partial_\mu g_{\rho\nu} + \partial_\nu g_{\rho\mu} - \partial_\rho g_{\mu\nu}\right)$$

$$R^\rho_{\ \sigma\mu\nu} = \partial_\mu\Gamma^\rho_{\nu\sigma} - \partial_\nu\Gamma^\rho_{\mu\sigma} + \Gamma^\rho_{\mu\lambda}\Gamma^\lambda_{\nu\sigma} - \Gamma^\rho_{\nu\lambda}\Gamma^\lambda_{\mu\sigma}$$

In [1]:
import sys; sys.path.insert(0, "/home/minjae/Minjae/IndexCalc")

import jax
import jax.numpy as jnp
import numpy as np
jax.config.update("jax_enable_x64", True)

from indexcalc import Metric, Coordinates

## 1. 2-Sphere: $ds^2 = r_0^2(d\theta^2 + \sin^2\theta\, d\varphi^2)$

Gaussian curvature $K = 1/r_0^2$, Ricci scalar $R = 2/r_0^2$.

In [2]:
r0 = 3.0

def sphere_metric(x):
    theta = x[0]
    return r0**2 * jnp.array([[1., 0.], [0., jnp.sin(theta)**2]])

g = Metric(sphere_metric, "spherical", dim=2)
print(f"좌표: {g.coords}")

result = g.at(jnp.array([jnp.pi/3, 0.5]))

print(f"\nChristoffel (비영 성분):")
result.show("christoffel")

R_analytic = 2.0 / r0**2
print(f"\nRicci scalar R = {float(result.R):.8f}  (해석: {R_analytic:.8f})")
print(f"Gaussian curvature K = {float(result.R)/2:.8f}  (해석: {1/r0**2:.8f})")

assert abs(float(result.R) - R_analytic) < 1e-10, "FAIL"
print("\n✓ 통과")

좌표: Coordinates(θ, φ | spherical)

Christoffel (비영 성분):
  Γ^θ_{φφ} = -0.43301270
  Γ^φ_{θφ} = 0.57735027

Ricci scalar R = 0.22222222  (해석: 0.22222222)
Gaussian curvature K = 0.11111111  (해석: 0.11111111)

✓ 통과


## 2. Schwarzschild: 진공해 검증

$$ds^2 = -\left(1-\frac{r_s}{r}\right)dt^2 + \frac{dr^2}{1-r_s/r} + r^2 d\Omega^2$$

진공 Einstein 방정식: $R_{\mu\nu} = 0$, Kretschner scalar: $K = 48M^2/r^6$.

In [3]:
M, rs = 1.0, 2.0

def schwarzschild(x):
    r, theta = x[1], x[2]
    f = 1 - rs / r
    return jnp.diag(jnp.array([-f, 1/f, r**2, r**2 * jnp.sin(theta)**2]))

g = Metric(schwarzschild, "spherical", signature=(3, 1))
print(f"좌표: {g.coords},  {g}")

r_eval = 5.0
result = g.at(jnp.array([0., r_eval, jnp.pi/2, 0.]))

print(f"\n|R_μν| max = {float(jnp.max(jnp.abs(result.Ric))):.2e}  (해석: 0)")
print(f"R = {float(result.R):.2e}  (해석: 0)")

K_analytic = 48 * M**2 / r_eval**6
print(f"Kretschner K = {float(result.K):.10f}  (해석: {K_analytic:.10f})")

print(f"\nChristoffel (비영):")
result.show("christoffel")

print(f"\nRiemann (비영):")
result.show("riemann")

print(f"\nsummary: {result.summary()}")

좌표: Coordinates(t, r, θ, φ | spherical),  Metric(Coordinates(t, r, θ, φ | spherical), signature=−+++)

|R_μν| max = 5.55e-17  (해석: 0)
R = -1.69e-17  (해석: 0)
Kretschner K = 0.0030720000  (해석: 0.0030720000)

Christoffel (비영):
  Γ^t_{tr} = 0.06666667
  Γ^r_{tt} = 0.02400000
  Γ^r_{rr} = -0.06666667
  Γ^r_{θθ} = -3.00000000
  Γ^r_{φφ} = -3.00000000
  Γ^θ_{rθ} = 0.20000000
  Γ^φ_{rφ} = 0.20000000

Riemann (비영):
  R^t_{rtr} = 0.02666667
  R^t_{θtθ} = -0.20000000
  R^t_{φtφ} = -0.20000000
  R^r_{ttr} = 0.00960000
  R^r_{θrθ} = -0.20000000
  R^r_{φrφ} = -0.20000000
  R^θ_{ttθ} = -0.00480000
  R^θ_{rrθ} = 0.01333333
  R^θ_{φθφ} = 0.40000000
  R^φ_{ttφ} = -0.00480000
  R^φ_{rrφ} = 0.01333333
  R^φ_{θθφ} = -0.40000000

summary: R = -1.69309e-17, K = 0.003072, |G|_max = 1.56125e-16


## 3. de Sitter: 우주상수 검증

$$R = 4\Lambda, \quad G_{\mu\nu} = -\Lambda\, g_{\mu\nu}$$

In [4]:
Lambda = 0.3

def de_sitter(x):
    r, theta = x[1], x[2]
    f = 1 - Lambda * r**2 / 3
    return jnp.diag(jnp.array([-f, 1/f, r**2, r**2 * jnp.sin(theta)**2]))

g = Metric(de_sitter, "spherical", signature=(3, 1))
result = g.at(jnp.array([0., 1.0, jnp.pi/2, 0.]))

print(f"R = {float(result.R):.8f}  (해석: {4*Lambda:.8f})")

residual = result.G + Lambda * result.g
print(f"|G_μν + Λg_μν| max = {float(jnp.max(jnp.abs(residual))):.2e}")

print(f"\nEinstein tensor:")
result.show("einstein")

assert jnp.allclose(residual, 0., atol=1e-6)
print("\n✓ Einstein 방정식 검증 완료")

R = 1.20000000  (해석: 1.20000000)
|G_μν + Λg_μν| max = 1.67e-16

Einstein tensor:
  G_{tt} = 0.27000000
  G_{rr} = -0.33333333
  G_{θθ} = -0.30000000
  G_{φφ} = -0.30000000

✓ Einstein 방정식 검증 완료


## 4. FLRW 우주론: $ds^2 = -dt^2 + a(t)^2(dx^2+dy^2+dz^2)$

$a(t) = e^{Ht}$ (de Sitter expansion) → $R = 12H^2$.

In [5]:
H = 0.5

def flrw_flat(x):
    t = x[0]
    a2 = jnp.exp(2 * H * t)
    return jnp.diag(jnp.array([-1., a2, a2, a2]))

g = Metric(flrw_flat, "cartesian", signature=(3, 1))
result = g.at(jnp.array([1.0, 0., 0., 0.]))

R_analytic = 12 * H**2
print(f"R = {float(result.R):.8f}  (해석: {R_analytic:.8f})")

print(f"\nChristoffel (비영):")
result.show("christoffel")

print(f"\nRicci tensor:")
result.show("ricci")

assert abs(float(result.R) - R_analytic) < 1e-6
print("\n✓ 통과")

R = 3.00000000  (해석: 3.00000000)

Christoffel (비영):
  Γ^t_{xx} = 1.35914091
  Γ^t_{yy} = 1.35914091
  Γ^t_{zz} = 1.35914091
  Γ^x_{tx} = 0.50000000
  Γ^y_{ty} = 0.50000000
  Γ^z_{tz} = 0.50000000

Ricci tensor:
  R_{tt} = -0.75000000
  R_{xx} = 2.03871137
  R_{yy} = 2.03871137
  R_{zz} = 2.03871137

✓ 통과


## 5. Coordinates API 확인

preset, custom, signature 조합 테스트.

In [6]:
# Preset 좌표계
print("Preset 좌표계:")
for system in ["cartesian", "spherical", "cylindrical"]:
    c3 = Coordinates.preset(system, dim=3)
    c4 = Coordinates.preset(system, signature=(3,1))
    print(f"  {system} dim=3: {c3}")
    print(f"  {system} sig=(3,1): {c4}")

# Custom 좌표
print("\nCustom 좌표:")
g_custom = Metric(lambda x: jnp.eye(3), ["u", "v", "w"])
print(f"  {g_custom}")
print(f"  signature: {g_custom.signature}")

# Explicit signature
print("\nExplicit signature:")
g_exp = Metric(lambda x: jnp.eye(4), ["t","x","y","z"], signature=(-1,1,1,1))
print(f"  {g_exp}")

# dim/signature 불일치 → 에러
print("\ndim/signature 불일치 테스트:")
try:
    Metric(lambda x: jnp.eye(3), ["a","b","c"], signature=(3,1))
except ValueError as e:
    print(f"  ✓ 에러 발생: {e}")

Preset 좌표계:
  cartesian dim=3: Coordinates(x, y, z | cartesian)
  cartesian sig=(3,1): Coordinates(t, x, y, z | cartesian)
  spherical dim=3: Coordinates(r, θ, φ | spherical)
  spherical sig=(3,1): Coordinates(t, r, θ, φ | spherical)
  cylindrical dim=3: Coordinates(ρ, φ, z | cylindrical)
  cylindrical sig=(3,1): Coordinates(t, ρ, φ, z | cylindrical)

Custom 좌표:
  Metric(Coordinates(u, v, w), signature=+++)
  signature: (1, 1, 1)

Explicit signature:
  Metric(Coordinates(t, x, y, z), signature=−+++)

dim/signature 불일치 테스트:
  ✓ 에러 발생: Signature (3, 1) sums to 4, but dim=3.
